In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from utils import CustomDataset,SelfAttention
import tiktoken

# Load variables and text

In [2]:
enc = tiktoken.get_encoding("cl100k_base")
print(f"{enc.n_vocab} tokens in the encoding")

with open("sample_text.txt", "r") as f:
    text = f.read()

enc_text = enc.encode(text)
print(f"Encoded text length: {len(enc_text)}")

100277 tokens in the encoding
Encoded text length: 304


# Code from self-attention notebook

In [29]:
dataloader = DataLoader(CustomDataset(text, enc, max_length=4, stride=4), batch_size=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
input_ids, target_ids = first_batch


vicab_size = enc.n_vocab
print(f"Vocab size: {vicab_size}") 

embed_dims = 8 
print(f"Output dims: {embed_dims}")

token_embed = torch.nn.Embedding(vicab_size, embed_dims)
print(f"Token embedding shape: {token_embed.weight.shape}")

inputs = token_embed(input_ids)
print(f"Input shape: {inputs.shape} -> batch_size x sequence_length x embedding_dim")

d_in = inputs.shape[-1]
d_out = 8 # dimensions of the output 

num_heads = 2
head_dim = d_out // num_heads
print(f"Head dim: {head_dim}")

print(f"d_in: {d_in}, d_out: {d_out}, num_heads: {num_heads}, head_dim: {head_dim}")

Vocab size: 100277
Output dims: 8
Token embedding shape: torch.Size([100277, 8])
Input shape: torch.Size([1, 4, 8]) -> batch_size x sequence_length x embedding_dim
Head dim: 4
d_in: 8, d_out: 8, num_heads: 2, head_dim: 4


In [30]:
Query_w = torch.nn.Linear(d_in, d_out)
Key_w = torch.nn.Linear(d_in, d_out)
Value_w = torch.nn.Linear(d_in, d_out)

print(f"Input shape: {inputs.shape}")
print(f"Query weight shape: {Query_w.weight.shape}")


Input shape: torch.Size([1, 4, 8])
Query weight shape: torch.Size([8, 8])


In [36]:
keys = Key_w(inputs)
query = Query_w(inputs)
values = Value_w(inputs)

batch_size, input_length, d_in = inputs.shape
print(f"Keys shape: {keys.shape} -> batch_size x sequence_length x embedding_dim")

print("Query :")
query

Keys shape: torch.Size([1, 4, 8]) -> batch_size x sequence_length x embedding_dim
Query :


tensor([[[ 0.1408, -0.5358, -0.8355, -0.4405,  0.3363, -0.7353, -0.5687,
          -0.0444],
         [-0.9810, -0.9555, -0.1738, -0.2963,  1.0731, -0.7478, -0.7273,
           0.2905],
         [-0.7366, -1.8912,  0.8053, -1.0600,  0.2214,  0.1501,  0.6863,
           0.3530],
         [ 0.1286,  0.7264,  0.1295, -0.0687,  0.6819,  0.6903, -0.5548,
           0.2665]]], grad_fn=<ViewBackward0>)

In [37]:
# (batch_size, sequence_length, d_out) -> (batch_size, sequence_length, num_heads, head_dim)
keys = keys.view(batch_size, input_length, num_heads, head_dim)
query = query.view(batch_size, input_length, num_heads, head_dim)
values = values.view(batch_size, input_length, num_heads, head_dim)


# (batch_size, sequence_length, num_heads, head_dim) -> (batch_size, num_heads, sequence_length, head_dim)
keys = keys.transpose(1, 2)
query = query.transpose(1, 2)
values = values.transpose(1, 2)

attention_scores = query @ keys.transpose(2,3)/(head_dim**0.5)
mask = torch.triu(torch.ones(input_length, input_length), diagonal=1).bool()
attention_scores = attention_scores.masked_fill(mask, float('-inf'))

attention_weights = torch.nn.functional.softmax(attention_scores, dim=-1)
attention_output = attention_weights @ values